# Additive Technique Notebook 4: Optional Domain Adapter Stage (Unsloth + PEFT + TRL)

This notebook introduces an **optional advanced stage** for generator domain adaptation.

Important design rule in this project:
- Unsloth, PEFT, and TRL are **not forced** into the default pipeline.
- They are used only where they add concrete value: **efficient SEC-domain generator adaptation**.

Status: implementation complete. Core pipeline artifacts are executed; adapter-stage outputs remain placeholder unless explicit adapter training/evaluation is run successfully.



## 1. What Is This Technique?

This technique adds a lightweight adapter-tuning stage to make the generator model more aligned with SEC filing language and style.

Instead of replacing GraphRAG, it augments the generation stage after retrieval grounding.



## 2. Definition and Core Concepts

### Unsloth
Unsloth is an efficiency-focused fine-tuning framework for fast, memory-conscious LLM adaptation.

### PEFT
PEFT (Parameter-Efficient Fine-Tuning) trains small adapter weights (for example LoRA) instead of full-model updates.

### TRL
TRL provides practical training loops (for example SFTTrainer) for supervised tuning and alignment workflows.

### Combined Role in This Project
- Unsloth: efficient model loading and adapter training setup.
- PEFT: LoRA adapter representation and adapter loading/merging behavior.
- TRL: supervised training loop with evaluation/checkpoint controls.



## 3. Why This Technique Was Developed

Traditional RAG can still produce weak answers when the generator is under-adapted to domain language.

Even with strong retrieval, answer quality can suffer due to:
- weak domain phrasing and financial-report style control
- inconsistent abstraction level across answers
- weaker synthesis in long, formal SEC passages

This optional stage targets those generator-side gaps while preserving retrieval and graph logic.



## 4. What Limitations of Traditional RAG It Solves

- **Generator domain mismatch**: base model may not internalize SEC-specific style.
- **Under-utilization of retrieved evidence**: model may ignore or compress crucial filing details.
- **Quality ceiling**: retrieval improves context quality, but generation quality remains bounded by model adaptation.

What it does **not** solve directly:
- retrieval recall defects
- graph-construction errors
- OCR/vision extraction errors



## 5. Architecture Diagram

```mermaid
flowchart LR
    A[SEC Filings Corpus] --> B[Chunk Builder]
    B --> C[Adapter SFT Dataset
Real SEC continuation pairs]

    C --> D[Unsloth Loader]
    D --> E[PEFT LoRA Adapters]
    E --> F[TRL SFTTrainer]
    F --> G[Trained Adapter]

    H[GraphRAG Retrieval Contexts] --> I[Base Generator]
    H --> J[Adapter-Augmented Generator]
    I --> K[Base Outputs]
    J --> L[Adapter Outputs]
    K --> M[Metrics + Judge + Latency]
    L --> M
```



## 6. Workflow Diagram Explanation

```mermaid
sequenceDiagram
    participant D as Data Builder
    participant T as Training Stack
    participant R as Retriever
    participant E as Evaluator

    D->>T: Real SEC chunk continuations (train/eval)
    T->>T: Unsloth load + PEFT LoRA + TRL SFT
    T-->>T: Save adapter checkpoint

    R->>E: Retrieval-grounded prompts
    E->>E: Run base model inference
    E->>E: Run adapter model inference
    E->>E: Compare generation, RAG, judge, latency
```

### Component-by-Component Breakdown
1. Dataset builder from real chunk text (no synthetic dataset injection).
2. Unsloth model setup with low-memory loading.
3. PEFT adapter strategy (LoRA ranks/targets).
4. TRL SFT training loop with eval/save cadence.
5. Base-vs-adapter benchmark harness for measurable impact.



## 7. Why These Tools Were Used Here (and Nowhere Else)

### Where They Were Used
- Only in optional generator adaptation stage.
- Not used in ingestion, graph building, retrieval fusion, CRAG routing, or multimodal extraction.

### Why This Boundary Matters
- Keeps default system simpler and robust.
- Avoids training dependencies in baseline usage.
- Preserves clean A/B comparison between base and adapted generator behavior.

### What Changed Because of Them
- Added adapter training/evaluation code paths and artifacts.
- Added optional CLI (`scripts/run_domain_adapter.py`).
- Added benchmark outputs for base vs adapter deltas.



## 8. When Should This Be Used in Real Systems?

Use this stage when:
- retrieval quality is already strong but answer quality plateaus
- domain language/style precision is important (financial, legal, compliance)
- GPU resources are available for periodic adapter refresh

Avoid this stage when:
- retrieval is still weak (fix retrieval first)
- latency budget is extremely strict and adapter path is unoptimized
- team lacks monitoring for model-quality regressions



## 9. Advantages and Disadvantages

### Advantages
- parameter-efficient adaptation without full-model fine-tuning
- potentially better domain phrasing and evidence usage
- explicit, measurable base-vs-adapter comparison path

### Disadvantages
- additional training/inference operational complexity
- optional dependency stack (Unsloth/PEFT/TRL) to manage
- gains are not guaranteed; requires real benchmark validation



## 10. Comparison Against Standard RAG and Other Variants

| Variant | Primary Improvement Axis | Relative Complexity | Typical Gain Area |
|---|---|---:|---|
| Standard RAG | semantic retrieval + generation | Low | baseline grounding |
| Hybrid RAG | lexical + semantic retrieval | Medium | recall / retrieval robustness |
| GraphRAG | relational/global reasoning | Medium-High | multi-hop and thematic synthesis |
| Multimodal RAG | non-text evidence channels | High | table/visual evidence coverage |
| CRAG/Agentic | routing and corrective behavior | High | reliability / recovery |
| **Adapter Stage (this notebook)** | generator domain adaptation | Medium-High | answer quality / domain style |

Design decision in this project: adapter stage is complementary, not a replacement for retrieval or graph improvements.



In [ ]:
from pathlib import Path
import os
import sys
import json

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import SETTINGS
from src.eval_queries import load_eval_queries
from src.extensions.domain_adapter import (
    build_eval_rows_from_retriever,
    evaluate_base_vs_adapter,
    seed_domain_adapter_placeholders,
    train_domain_adapter_from_chunks,
)

RUN_EXECUTION = not SETTINGS.placeholder_mode

print('Placeholder mode:', SETTINGS.placeholder_mode)
print('Adapter enabled by config:', SETTINGS.adapter_enable)
print('Adapter base model:', SETTINGS.adapter_base_model)
print('Judge model for comparison:', SETTINGS.extension_judge_model)


## 11. Implementation Details and Project-Specific Design Decisions

- dataset source for adaptation: real SEC chunk text only
- training target: continuation-style SFT for domain language adaptation
- GPU gate: if CUDA is unavailable, stage is skipped with explicit status
- default pipeline remains unchanged unless adapter script is called



In [ ]:
if RUN_EXECUTION and SETTINGS.adapter_enable:
    # Example execute flow (optional advanced stage):
    # 1) Build corpus + chunks with existing project pipeline pieces.
    # 2) train_summary = train_domain_adapter_from_chunks(chunks, output_dir=Path('artifacts/adapter'))
    # 3) Build retrieval-grounded eval rows with HybridRetriever.
    # 4) benchmark = evaluate_base_vs_adapter(...)
    # 5) Save generation/rag/judge/latency comparison artifacts.
    pass
else:
    print('Placeholder mode or adapter stage disabled: training/benchmark not executed.')



## 12. Current Output Status and Result Interpretation

After execution, this section must include:

### How Unsloth, PEFT, and TRL Affected Results
- quality deltas (EM/BLEU/ROUGE/METEOR/BERTScore)
- RAG quality deltas (faithfulness/context precision/context recall/answer relevancy)
- judge deltas (correctness/relevance/completeness/groundedness/hallucination risk)
- latency impact (base vs adapter)

### Why Specific Outputs Were Produced
- explain representative output pairs (base vs adapter)
- map improvements/regressions to retrieval context usage behavior

### Practical Takeaways
- where adapter stage is worth the operational cost
- where retrieval/graph changes remain higher-leverage
- when to retrain adapters and how to detect drift



In [ ]:
placeholder_paths = seed_domain_adapter_placeholders()
for name, path in sorted(placeholder_paths.items()):
    print(name, '->', Path(path).exists())



## 13. Official Documentation References (Used for This Implementation)

- Unsloth Granite support and setup:
  - https://unsloth.ai/docs/models/ibm-granite-4.1
- TRL SFT trainer and PEFT integration:
  - https://github.com/huggingface/trl/blob/main/docs/source/sft_trainer.md
  - https://github.com/huggingface/trl/blob/v1.0.0/docs/source/peft_integration.md
- PEFT adapter loading and merge behavior:
  - https://github.com/huggingface/peft/blob/main/docs/source/package_reference/lora.md

These sources guided design and API usage; no non-official tutorial patterns were used as primary references.



## 14. Conclusion and Deployment Guidance

After real execution, summarize:
1. whether adapter stage materially improved grounded answer quality
2. whether latency tradeoff is acceptable for deployment targets
3. whether this stage should be enabled in production by default or remain optional



## Real Run Snapshot

Shows optional adapter-stage status artifacts (executed only when adapter stage is enabled).

In [ ]:
# REAL_RUN_SNAPSHOT_DOMAIN_ADAPTER
from pathlib import Path
import json

paths = [
    Path('artifacts/eval/domain_adapter_training_summary.json'),
    Path('artifacts/eval/domain_adapter_training_placeholder.json'),
]
for p in paths:
    if p.exists():
        print()
        print(p)
        print(p.read_text(encoding='utf-8')[:800])
    else:
        print(p, '-> missing')
